# CC2 — Plano de Preparação dos Dados
**SIMA – Sistema Inteligente de Monitoramento e Alerta de Alagamentos**  
Cesar School · 5º Período CC · Trilha de Análise e Visualização de Dados

---

Este notebook documenta o **plano de preparação dos dados** do SIMA: fontes utilizadas, estratégia de limpeza, variáveis derivadas, composição dos datasets por notebook e decisões de pré-processamento com justificativas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# Paleta semântica SIMA
VERDE = '#10b981'
AMBAR = '#f59e0b'
VERM  = '#ef4444'
AZUL  = '#1d4ed8'
CINZA = '#64748b'
PRETO = '#0f172a'
FUNDO = '#f8fafc'

plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         False,
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
})

## 1. Fontes de Dados

A análise combina quatro fontes:

| # | Fonte | Tipo | Formato | Origem |
|---|-------|------|---------|--------|
| F1 | Relatos do SIMA | Própria | CSV exportado do PostgreSQL | Tabela `relatos` |
| F2 | Bairros de Recife | Própria | CSV exportado do PostgreSQL | Tabela `bairros` |
| F3 | Questionário de usuários | Primária | CSV do Google Forms | 42 respostas |
| F4 | Relatos simulados | Simulada | Gerado em Python | Calibrado nas proporções reais |

**Por que dados simulados?** O banco real conta com apenas 41 relatos desde o lançamento do MVP — volume insuficiente para regressão e classificação, que exigem mínimo de 50–100 amostras por classe. Os dados simulados são gerados com as mesmas proporções e sazonalidade dos dados reais.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.5))
ax.set_xlim(0, 13); ax.set_ylim(0, 4); ax.axis('off')

fontes = [
    (0.3,  AZUL,  'F1', 'Relatos SIMA',    'relatos.csv\n41 registros reais'),
    (3.1,  VERDE, 'F2', 'Bairros Recife',  'bairros.csv\n94 bairros oficiais'),
    (5.9,  AMBAR, 'F3', 'Questionário',    'Google Forms\n42 respostas'),
    (8.7,  CINZA, 'F4', 'Simulados',       'Script Python\n400–800 registros'),
]

for x, cor, num, titulo, desc in fontes:
    ax.add_patch(FancyBboxPatch((x, 0.6), 2.5, 2.8,
                                boxstyle='round,pad=0.12',
                                facecolor=cor, edgecolor='none', alpha=0.10))
    ax.add_patch(FancyBboxPatch((x, 2.8), 2.5, 0.6,
                                boxstyle='round,pad=0.06',
                                facecolor=cor, edgecolor='none', alpha=0.80))
    ax.text(x + 1.25, 3.1, f'{num} — {titulo}',
            ha='center', va='center', fontsize=9.5, fontweight='bold', color='white')
    ax.text(x + 1.25, 1.8, desc,
            ha='center', va='center', fontsize=9.5, color=PRETO, multialignment='center')

# Seta para merge
ax.annotate('', xy=(11.4, 2.0), xytext=(10.9, 2.0),
            arrowprops=dict(arrowstyle='->', color=CINZA, lw=1.8))
ax.add_patch(FancyBboxPatch((11.4, 1.3), 1.3, 1.4,
                             boxstyle='round,pad=0.08',
                             facecolor=VERM, edgecolor='none', alpha=0.15))
ax.text(12.05, 2.05, 'MERGE', ha='center', va='center',
        fontsize=10, fontweight='bold', color=VERM)
ax.text(12.05, 1.62, '→ df_full', ha='center', va='center',
        fontsize=9, color=CINZA)

ax.set_title('Fontes de Dados do SIMA', fontsize=13,
             fontweight='bold', color=PRETO, pad=10)
plt.tight_layout()
plt.show()

## 2. Composição dos Datasets por Notebook

Cada notebook usa um corte diferente dos dados, com justificativa explícita para uso de simulados.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

notebooks = [
    ('CC3 — EDA',           41,  400, 441),
    ('CC4 — Regressão',      0,  800, 800),
    ('CC5 — Classificadores', 0, 800, 800),
]

for ax, (nome, reais, sim, total) in zip(axes, notebooks):
    if reais > 0:
        ax.pie(
            [reais, sim],
            labels=[f'Reais ({reais})', f'Simulados ({sim})'],
            colors=[AZUL, '#bfdbfe'],
            autopct='%1.1f%%',
            startangle=90,
            explode=(0.06, 0),
            wedgeprops={'edgecolor': 'white', 'linewidth': 3},
            textprops={'fontsize': 10},
        )
    else:
        ax.pie(
            [sim],
            labels=[f'Simulados ({sim})'],
            colors=['#bfdbfe'],
            autopct='%1.1f%%',
            startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 3},
            textprops={'fontsize': 10},
        )
    ax.set_title(f'{nome}\n(n={total})', fontsize=11, fontweight='bold', color=PRETO)

plt.suptitle('Composição dos Datasets por Notebook', fontsize=13,
             fontweight='bold', color=PRETO, y=1.02)
plt.tight_layout(pad=2)
plt.show()

print('CC3 mantém os 41 relatos reais visíveis + 400 simulados, exibindo análises lado a lado.')
print('CC4 e CC5 usam apenas simulados: 41 registros não atingem o mínimo por classe.')

## 3. Estratégia de Limpeza

Cinco problemas foram identificados nos dados brutos e tratados conforme a tabela abaixo.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.axis('off')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title('Problemas de Qualidade de Dados e Tratamentos', fontsize=12,
             fontweight='bold', color=PRETO, pad=12)

problemas = [
    (VERM,  'bairro_id nulo',           'Relatos sem bairro',
     'Mantidos em análises por nível/tempo;\nexcluídos de análises geográficas'),
    (AMBAR, 'Duplicatas local+tempo',   'Múltiplos envios do mesmo evento',
     'Raio 50m + janela 5 min;\nmais recente descartado'),
    (CINZA, 'Datas fora do período',    'created_at antes do lançamento',
     'Filtro de data:\napenas período operacional'),
    (VERDE, 'NaN no questionário',      'Perguntas opcionais sem resposta',
     'NaN excluídos por pergunta\nantes de calcular frequências'),
    (AZUL,  'nivel inconsistente',      'Valores fora do vocabulário',
     "Filtro: apenas\n{'baixo', 'medio', 'alto'}"),
]

ncols = len(problemas)
w = 1 / ncols

for i, (cor, titulo, ocorrencia, tratamento) in enumerate(problemas):
    x0 = i * w + 0.01
    # Card
    ax.add_patch(FancyBboxPatch((x0, 0.04), w - 0.02, 0.92,
                                boxstyle='round,pad=0.01',
                                facecolor=cor, edgecolor='none', alpha=0.07))
    # Header colorido
    ax.add_patch(FancyBboxPatch((x0, 0.78), w - 0.02, 0.18,
                                boxstyle='round,pad=0.005',
                                facecolor=cor, edgecolor='none', alpha=0.75))
    ax.text(x0 + (w - 0.02) / 2, 0.87, titulo,
            ha='center', va='center', fontsize=8.5, fontweight='bold', color='white',
            multialignment='center')
    # Ocorrência
    ax.text(x0 + (w - 0.02) / 2, 0.63, 'Ocorrência:', ha='center', va='center',
            fontsize=7.5, color=CINZA, fontweight='bold')
    ax.text(x0 + (w - 0.02) / 2, 0.50, ocorrencia, ha='center', va='center',
            fontsize=8.5, color=PRETO, multialignment='center')
    # Divisor
    ax.plot([x0 + 0.01, x0 + w - 0.03], [0.38, 0.38], color='#e2e8f0', lw=1)
    # Tratamento
    ax.text(x0 + (w - 0.02) / 2, 0.31, 'Tratamento:', ha='center', va='center',
            fontsize=7.5, color=cor, fontweight='bold')
    ax.text(x0 + (w - 0.02) / 2, 0.17, tratamento, ha='center', va='center',
            fontsize=8.5, color=PRETO, multialignment='center')

plt.tight_layout(pad=1)
plt.show()

## 4. Variáveis Derivadas de `created_at`

A coluna `created_at` é a principal fonte de features temporais. O diagrama abaixo mostra as transformações aplicadas.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.5))
ax.set_xlim(0, 13); ax.set_ylim(0, 4); ax.axis('off')

# Caixa central: created_at
ax.add_patch(FancyBboxPatch((5.2, 1.3), 2.6, 1.4,
                             boxstyle='round,pad=0.12',
                             facecolor=AZUL, edgecolor='none', alpha=0.85))
ax.text(6.5, 2.0, 'created_at\n(TIMESTAMP)',
        ha='center', va='center', fontsize=11, fontweight='bold', color='white',
        multialignment='center')

derivadas = [
    (0.5,  VERM,  'hora',  '.dt.hour',      'Padrão\nintradiário\n(16h–23h pico)'),
    (3.2,  AMBAR, 'mês',   '.dt.month',     'Sazonalidade\n(mar–ago\nchuvoso)'),
    (9.8,  VERDE, 'dia',   '.dt.dayofyear', 'Índice para\nregressão\ntemporal'),
    (12.4, CINZA, 'data',  '.dt.date',      'Agrupamento\npor volume\ndiário'),
]

for x, cor, nome, formula, just in derivadas:
    # Caixa derivada
    ax.add_patch(FancyBboxPatch((x, 1.45), 2.4, 1.1,
                                boxstyle='round,pad=0.08',
                                facecolor=cor, edgecolor='none', alpha=0.12))
    ax.add_patch(FancyBboxPatch((x, 2.25), 2.4, 0.30,
                                boxstyle='round,pad=0.04',
                                facecolor=cor, edgecolor='none', alpha=0.70))
    ax.text(x + 1.2, 2.40, nome, ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')
    ax.text(x + 1.2, 1.88, formula, ha='center', va='center',
            fontsize=9, color=PRETO, fontfamily='monospace')
    # Texto de justificativa abaixo
    ax.text(x + 1.2, 0.90, just, ha='center', va='center',
            fontsize=8, color=CINZA, multialignment='center')
    # Seta
    if x < 5.2:
        ax.annotate('', xy=(x + 2.4, 2.0), xytext=(5.2, 2.0),
                    arrowprops=dict(arrowstyle='<-', color=cor, lw=1.6))
    else:
        ax.annotate('', xy=(x, 2.0), xytext=(7.8, 2.0),
                    arrowprops=dict(arrowstyle='->', color=cor, lw=1.6))

ax.set_title('Variáveis Derivadas de created_at', fontsize=12,
             fontweight='bold', color=PRETO, pad=8)
plt.tight_layout()
plt.show()

## 5. Codificação de Variáveis Categóricas

Duas variáveis categóricas precisam ser convertidas para numéricas antes dos modelos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# nivel → nivel_num (ordinal) e nivel_enc (LabelEncoder)
ax = axes[0]
ax.axis('off'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title('Codificação de nivel', fontsize=11, fontweight='bold', color=PRETO)

mapeamentos = [
    ('baixo',  VERDE, 'nivel_num = 1', 'nivel_enc = 1', '(ord. alfabética: baixo=1)'),
    ('medio',  AMBAR, 'nivel_num = 2', 'nivel_enc = 2', '(ord. alfabética: medio=2)'),
    ('alto',   VERM,  'nivel_num = 3', 'nivel_enc = 0', '(ord. alfabética: alto=0)'),
]

for i, (nivel, cor, num_str, enc_str, nota) in enumerate(mapeamentos):
    y = 0.72 - i * 0.28
    # Nível original
    ax.add_patch(FancyBboxPatch((0.02, y), 0.18, 0.20,
                                boxstyle='round,pad=0.01',
                                facecolor=cor, edgecolor='none', alpha=0.75))
    ax.text(0.11, y + 0.10, nivel, ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')
    # Seta
    ax.annotate('', xy=(0.25, y + 0.10), xytext=(0.20, y + 0.10),
                arrowprops=dict(arrowstyle='->', color=CINZA, lw=1.2))
    # nivel_num
    ax.add_patch(FancyBboxPatch((0.26, y), 0.22, 0.20,
                                boxstyle='round,pad=0.01',
                                facecolor=cor, edgecolor='none', alpha=0.15))
    ax.text(0.37, y + 0.10, num_str, ha='center', va='center',
            fontsize=9, color=PRETO)
    # nivel_enc
    ax.add_patch(FancyBboxPatch((0.52, y), 0.26, 0.20,
                                boxstyle='round,pad=0.01',
                                facecolor=AZUL, edgecolor='none', alpha=0.10))
    ax.text(0.65, y + 0.10, enc_str, ha='center', va='center',
            fontsize=9, color=PRETO)
    ax.text(0.82, y + 0.10, nota, ha='left', va='center',
            fontsize=7.5, color=CINZA)

ax.text(0.11, 0.97, 'Original', ha='center', va='center',
        fontsize=8.5, color=CINZA, fontweight='bold')
ax.text(0.37, 0.97, 'nivel_num', ha='center', va='center',
        fontsize=8.5, color=CINZA, fontweight='bold')
ax.text(0.65, 0.97, 'nivel_enc', ha='center', va='center',
        fontsize=8.5, color=AZUL, fontweight='bold')
ax.text(0.37, 0.07, 'Ordinal (1/2/3): preserva ordem natural → regressão',
        ha='center', fontsize=8, color=CINZA, style='italic')
ax.text(0.65, 0.01, 'LabelEncoder (0/1/2): para classificação multiclasse',
        ha='center', fontsize=8, color=AZUL, style='italic')

# bairro → bairro_enc
ax2 = axes[1]
ax2.axis('off'); ax2.set_xlim(0, 1); ax2.set_ylim(0, 1)
ax2.set_title('Codificação de bairro', fontsize=11, fontweight='bold', color=PRETO)

bairros_ex = [
    ('Boa Viagem',  AZUL, 12),
    ('Ibura',       AZUL, 38),
    ('Imbiribeira', AZUL, 41),
    ('Pina',        AZUL, 67),
    ('... (94)',    CINZA, '...'),
]

for i, (nome, cor, enc) in enumerate(bairros_ex):
    y = 0.76 - i * 0.16
    ax2.add_patch(FancyBboxPatch((0.02, y), 0.38, 0.13,
                                  boxstyle='round,pad=0.01',
                                  facecolor=cor, edgecolor='none', alpha=0.10))
    ax2.text(0.21, y + 0.065, nome, ha='center', va='center',
             fontsize=9.5, color=PRETO)
    ax2.annotate('', xy=(0.48, y + 0.065), xytext=(0.40, y + 0.065),
                 arrowprops=dict(arrowstyle='->', color=CINZA, lw=1.2))
    ax2.add_patch(FancyBboxPatch((0.49, y), 0.16, 0.13,
                                  boxstyle='round,pad=0.01',
                                  facecolor=AZUL, edgecolor='none', alpha=0.15))
    ax2.text(0.57, y + 0.065, str(enc), ha='center', va='center',
             fontsize=10, color=AZUL, fontweight='bold')

ax2.text(0.21, 0.97, 'bairro (string)', ha='center', fontsize=8.5,
         color=CINZA, fontweight='bold')
ax2.text(0.57, 0.97, 'bairro_enc', ha='center', fontsize=8.5,
         color=AZUL, fontweight='bold')

ax2.add_patch(FancyBboxPatch((0.02, 0.04), 0.95, 0.10,
                              boxstyle='round,pad=0.01',
                              facecolor='#fef3c7', edgecolor=AMBAR,
                              linewidth=1, alpha=0.7))
ax2.text(0.50, 0.09, 'LabelEncoder em vez de One-Hot: 94 bairros gerariam\n'
         'matriz esparsa e overfitting. Adequado para árvore de decisão.',
         ha='center', va='center', fontsize=8, color=PRETO, multialignment='center')

plt.tight_layout(pad=2)
plt.show()

## 6. Variáveis de Engajamento (confirmações e denúncias)

Relatos reais têm contagens via tabelas `confirmacoes` e `denuncias` no banco. Nos dados simulados, são geradas com distribuição de Poisson calibrada por nível de severidade.

In [ ]:
np.random.seed(42)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribuições Poisson por nível
for ax, (var, lambdas, titulo) in zip(axes, [
    ('confirmacoes', {'baixo': 1, 'medio': 2, 'alto': 4},
     'Confirmações por Nível — Poisson(λ)'),
    ('denuncias',    {'baixo': 1.2, 'medio': 0.4, 'alto': 0.4},
     'Denúncias por Nível — Poisson(λ)'),
]):
    cores_nivel = {'baixo': VERDE, 'medio': AMBAR, 'alto': VERM}
    x_max = 15 if var == 'confirmacoes' else 6
    x_vals = np.arange(0, x_max + 1)

    for nivel, lam in lambdas.items():
        # P(X=k) Poisson
        probs = np.exp(-lam) * (lam ** x_vals) / np.array(
            [float(np.math.factorial(k)) for k in x_vals]
        )
        ax.plot(x_vals, probs, 'o-', color=cores_nivel[nivel],
                lw=2.5, ms=6, label=f'{nivel} (λ={lam})')

    ax.set_xlabel('Contagem')
    ax.set_ylabel('Probabilidade')
    ax.set_title(titulo, fontsize=11, fontweight='bold', color=PRETO)
    ax.legend(fontsize=9, frameon=False)
    ax.spines['left'].set_color('#e2e8f0')
    ax.spines['bottom'].set_color('#e2e8f0')

plt.tight_layout(pad=2)
plt.show()

print('Insight: Relatos de nível ALTO recebem em média 4× mais confirmações.')
print('Relatos de nível BAIXO recebem 3× mais denúncias — proxy de ruído/falso positivo.')

## 7. Features por Modelo

Cada notebook usa um subconjunto diferente de variáveis.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

modelos_features = [
    (
        'CC4 — Regressão',
        AMBAR,
        ['dia', 'nivel_num', 'hora', 'mes'],
        'confirmacoes',
        ['Contínua (série)', 'Ordinal 1–3', 'Discreta 0–23', 'Discreta 1–12'],
    ),
    (
        'CC5 — Classificadores',
        VERM,
        ['hora', 'mes', 'bairro_enc', 'confirmacoes', 'denuncias'],
        'nivel_enc',
        ['Discreta 0–23', 'Discreta 1–12', 'Inteiro 0–93', 'Poisson(λ=1-4)', 'Poisson(λ=0.4-1.2)'],
    ),
]

for ax, (nome, cor, features, target, tipos) in zip(axes, modelos_features):
    ax.axis('off'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_title(nome, fontsize=11, fontweight='bold', color=PRETO, pad=10)

    n = len(features)
    h = 0.82 / (n + 1.5)

    # Header
    ax.text(0.20, 0.96, 'Feature', fontsize=8.5, fontweight='bold',
            ha='center', va='center', color=CINZA)
    ax.text(0.65, 0.96, 'Tipo', fontsize=8.5, fontweight='bold',
            ha='center', va='center', color=CINZA)
    ax.text(0.92, 0.96, 'Papel', fontsize=8.5, fontweight='bold',
            ha='center', va='center', color=CINZA)

    for i, (feat, tipo) in enumerate(zip(features, tipos)):
        y = 0.88 - i * (h + 0.01)
        bg = FUNDO if i % 2 == 0 else 'white'
        ax.add_patch(mpatches.Rectangle((0.01, y - 0.005), 0.98, h,
                                         facecolor=bg, edgecolor='none'))
        ax.add_patch(FancyBboxPatch((0.02, y), 0.36, h * 0.80,
                                    boxstyle='round,pad=0.005',
                                    facecolor=cor, edgecolor='none', alpha=0.12))
        ax.text(0.20, y + h * 0.40, feat, ha='center', va='center',
                fontsize=9.5, color=PRETO, fontfamily='monospace')
        ax.text(0.65, y + h * 0.40, tipo, ha='center', va='center',
                fontsize=9, color=CINZA)
        ax.text(0.92, y + h * 0.40, 'X', ha='center', va='center',
                fontsize=10, color=cor, fontweight='bold')

    # Target
    yt = 0.88 - n * (h + 0.01) - 0.01
    ax.add_patch(FancyBboxPatch((0.01, yt), 0.98, h * 0.85,
                                boxstyle='round,pad=0.005',
                                facecolor=cor, edgecolor='none', alpha=0.20))
    ax.text(0.20, yt + h * 0.42, target, ha='center', va='center',
            fontsize=10, fontweight='bold', color=cor, fontfamily='monospace')
    ax.text(0.65, yt + h * 0.42,
            'Numérica' if 'num' in target or 'enc' not in target
            else 'Inteiro (encoded)',
            ha='center', va='center', fontsize=9, color=CINZA)
    ax.text(0.92, yt + h * 0.42, 'y', ha='center', va='center',
            fontsize=11, fontweight='bold', color=cor)

plt.tight_layout(pad=2)
plt.show()

## 8. Decisões de Pré-processamento

Cada decisão foi tomada com uma alternativa analisada e descartada.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.axis('off'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title('Decisões de Pré-processamento', fontsize=13,
             fontweight='bold', color=PRETO, pad=12)

decisoes = [
    ('Simulação direta',     'SMOTE',
     'Controla sazonalidade horária/mensal\nque SMOTE não consegue replicar'),
    ('LabelEncoder p/ bairro', 'One-Hot (94 colunas)',
     '94 dummies → matriz esparsa\ne overfitting no classificador'),
    ('nivel_num ordinal (1/2/3)', 'One-Hot p/ nivel',
     'Severidade tem ordem natural;\nordinal preserva essa semântica'),
    ('Janela de 1 ano (2024)', '6 meses ou 2 anos',
     '1 ano cobre ciclo completo:\nperíodo chuvoso + seco'),
    ('Poisson p/ confirmações', 'Distribuição uniforme',
     'Contagens discretas de eventos raros;\nλ calibrado por nível preserva correlação'),
]

row_h = 1 / len(decisoes)

# Cabeçalho
for x, txt, col in [(0.15, 'DECISÃO ESCOLHIDA', VERDE),
                     (0.51, 'ALTERNATIVA DESCARTADA', VERM),
                     (0.84, 'JUSTIFICATIVA', AZUL)]:
    ax.text(x, 0.97, txt, ha='center', va='center', fontsize=9,
            fontweight='bold', color=col)

ax.plot([0, 1], [0.935, 0.935], color='#e2e8f0', lw=1.5)

for i, (escolha, alt, just) in enumerate(decisoes):
    y_bot = 1 - (i + 1) * row_h
    y_mid = y_bot + row_h / 2
    bg = '#f8fafc' if i % 2 == 0 else 'white'
    ax.add_patch(mpatches.Rectangle((0, y_bot), 1, row_h - 0.01,
                                     facecolor=bg, edgecolor='none'))

    # Escolha
    ax.add_patch(FancyBboxPatch((0.01, y_bot + 0.02), 0.28, row_h - 0.05,
                                boxstyle='round,pad=0.005',
                                facecolor=VERDE, edgecolor='none', alpha=0.12))
    ax.text(0.15, y_mid, escolha, ha='center', va='center',
            fontsize=9.5, color=PRETO, multialignment='center')

    # Separador
    ax.text(0.37, y_mid, 'vs.', ha='center', va='center',
            fontsize=10, color=CINZA, fontweight='bold')

    # Alternativa
    ax.add_patch(FancyBboxPatch((0.40, y_bot + 0.02), 0.24, row_h - 0.05,
                                boxstyle='round,pad=0.005',
                                facecolor=VERM, edgecolor='none', alpha=0.08))
    ax.text(0.51, y_mid, alt, ha='center', va='center',
            fontsize=9, color=CINZA, style='italic', multialignment='center')

    # Justificativa
    ax.text(0.84, y_mid, just, ha='center', va='center',
            fontsize=8.5, color=PRETO, multialignment='center')

plt.tight_layout(pad=1)
plt.show()

## 9. Pipeline de Pré-processamento Completo

Visão end-to-end do fluxo de dados desde a extração até os datasets prontos.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.set_xlim(0, 14); ax.set_ylim(0, 5); ax.axis('off')

etapas = [
    (0.2,  AZUL,  '1. Extração',     'PostgreSQL\n→ relatos.csv\n→ bairros.csv'),
    (2.5,  AMBAR, '2. Limpeza',      'Remove nulos\nDuplicas\nDatas inválidas'),
    (4.8,  CINZA, '3. Merge',        'relatos\nLEFT JOIN\nbairros'),
    (7.1,  VERDE, '4. Derivação',    'hora, mes, dia\nnivel_num\nnivel_enc, bairro_enc'),
    (9.4,  AMBAR, '5. Simulação',    'Poisson para\nconfirmações\ne denúncias'),
    (11.7, VERM,  '6. Split',        'Train 80%\nTest 20%\nEstratificado'),
]

for x, cor, titulo, desc in etapas:
    ax.add_patch(FancyBboxPatch((x, 1.0), 2.0, 3.0,
                                boxstyle='round,pad=0.12',
                                facecolor=cor, edgecolor='none', alpha=0.08))
    ax.add_patch(FancyBboxPatch((x, 3.5), 2.0, 0.5,
                                boxstyle='round,pad=0.06',
                                facecolor=cor, edgecolor='none', alpha=0.75))
    ax.text(x + 1.0, 3.75, titulo, ha='center', va='center',
            fontsize=9, fontweight='bold', color='white')
    ax.text(x + 1.0, 2.25, desc, ha='center', va='center',
            fontsize=9, color=PRETO, multialignment='center')
    if x < 11.7:
        ax.annotate('', xy=(x + 2.2, 2.5), xytext=(x + 2.0, 2.5),
                    arrowprops=dict(arrowstyle='->', color=CINZA, lw=2))

# Outputs finais
for y, cor, txt in [
    (0.65, AZUL,  'df_full CC3 (n=441)'),
    (0.35, AMBAR, 'df CC4/CC5 (n=800)'),
]:
    ax.add_patch(FancyBboxPatch((11.2, y - 0.10), 2.5, 0.25,
                                boxstyle='round,pad=0.01',
                                facecolor=cor, edgecolor='none', alpha=0.15))
    ax.text(12.45, y + 0.025, txt, ha='center', va='center',
            fontsize=8.5, color=PRETO)

ax.annotate('', xy=(13.2, 0.6), xytext=(13.7, 0.9),
            arrowprops=dict(arrowstyle='-', color=CINZA, lw=0.8, linestyle='dashed'))

ax.set_title('Pipeline de Pré-processamento — SIMA', fontsize=13,
             fontweight='bold', color=PRETO, pad=10)
plt.tight_layout()
plt.show()

---

## Síntese

| Item | Detalhe |
|------|---------|
| Fontes de dados | 4 (relatos reais, bairros, questionário, simulados) |
| Volume real | 41 relatos exportados do PostgreSQL |
| Volume CC3 | 441 (41 reais + 400 simulados) |
| Volume CC4/CC5 | 800 (simulados calibrados) |
| Problemas de limpeza | 5 identificados e tratados |
| Variáveis derivadas de created_at | 4 (hora, mes, dia, data) |
| Codificações | nivel_num (ordinal), nivel_enc (LabelEncoder), bairro_enc (LabelEncoder) |
| Variáveis de engajamento | confirmacoes e denuncias — geradas com distribuição Poisson calibrada por nível |
| Decisões de pré-processamento | 5, todas com alternativa analisada e justificativa documentada |